# Learning Boolean gates inside an exact token machine

Linear lambda calculus is PTIME-complete: Mairson (JFP 2002) encodes
Boolean logic as linear terms, so any circuit becomes a term whose normal
form is its output. This notebook builds the geometry-of-interaction
token machine for linear terms as a **fixed, parameter-free function** —
auxiliary entry pushes a one-hot of its port, principal entry pops and
routes — and shows that Böhm-tree readback is then *exact by
construction* on constant-less maps. All the learning concentrates in the
constants: the gates `xor, and, or, nand, nor, mux` are opaque boxes whose
required behaviour, by GoI compositionality, is their reference term's own
execution formula — a plain function from incoming token stack to outgoing
token stack, learnable by supervised regression in seconds of JAX.

The result: the trained boxes decode every truth table exactly, compose
into never-seen half adders exactly, and — pointing the decoder at
`λp.λq.gate(p)(q)` — the machine **reads back the Mairson lambda term of
every gate from its learned weights**, node by node.

In [1]:
import random
import time

import jax
import jax.numpy as jnp
import numpy as np

from discopy.closed import Abstraction, BohmTree, Unitype, Variable

o = Unitype()
tree = BohmTree.from_term

## Mairson's linear encoding

Booleans are pairs — the first component is the value, the second is
garbage that preserves linearity — and the garbage collector `gid` turns a
boolean into the identity. Every bound variable occurs exactly once, so
the maps are pure gamma: `to_map` produces only `@` and `λ` nodes.

In [2]:
true = o(lambda x: o(lambda y: o(lambda z: z(x)(y))))
false = o(lambda x: o(lambda y: o(lambda z: z(y)(x))))
pair = o(lambda x: o(lambda y: o(lambda z: z(x)(y))))
i = o(lambda x: x)
gid = o(lambda b: b(i)(i)(i))
collect = o(lambda u: o(lambda v: gid(v)(u)))
not_ = o(lambda p: o(lambda x: o(lambda y: p(y)(x))))
and_ref = o(lambda p: o(lambda q: p(q)(false)(collect)))
or_ref = o(lambda p: o(lambda q: p(true)(q)(collect)))
copy = o(lambda p: p(pair(true)(true))(pair(false)(false))(o(
    lambda u: o(lambda v: u(o(lambda u1: o(lambda u2: v(o(
        lambda v1: o(lambda v2: pair(gid(v1)(u1))(gid(v2)(u2))))))))))))
xor_ref = o(lambda p: o(lambda q: copy(q)(o(lambda q1: o(lambda q2:
    p(not_(q1))(q2)(collect))))))
nand_ref = o(lambda p: o(lambda q: not_(and_ref(p)(q))))
nor_ref = o(lambda p: o(lambda q: not_(or_ref(p)(q))))
mux_ref = o(lambda s: o(lambda a: o(lambda b: s(a)(b)(collect))))

bits = {0: false, 1: true}
GATES = {
    "xor": (o("xor"), xor_ref, 2, lambda a, b: a ^ b),
    "and": (o("and"), and_ref, 2, lambda a, b: a & b),
    "or": (o("or"), or_ref, 2, lambda a, b: a | b),
    "nand": (o("nand"), nand_ref, 2, lambda a, b: 1 - (a & b)),
    "nor": (o("nor"), nor_ref, 2, lambda a, b: 1 - (a | b)),
    "mux": (o("mux"), mux_ref, 3, lambda s, a, b: a if s else b)}


def inputs_of(arity):
    if arity == 2:
        return [(a, b) for a in (0, 1) for b in (0, 1)]
    return [(s, a, b) for s in (0, 1) for a in (0, 1) for b in (0, 1)]


def apply_all(head, args):
    for arg in args:
        head = head(arg)
    return head


for name, (_, ref, arity, op) in GATES.items():
    for args in inputs_of(arity):
        term = apply_all(ref, [bits[bit] for bit in args])
        assert tree(term) == tree(bits[op(*args)]), (name, args)
    assert "δ" not in {box.name for box in ref.to_map().boxes}
print("truth tables of the six reference gates: OK")

truth tables of the six reference gates: OK


## The exact token machine

One token, one stack. The two rules are fixed and parameter-free:

* entering a box through an **auxiliary** port pushes the symbol of that
  port and exits through the principal;
* entering through the **principal** pops the top symbol and routes the
  rest out of the auxiliary it names — *by index, whichever kind pushed
  it*: a beta redex is a principal-principal cut, so `@`-pushed symbols
  are popped by `λ` boxes, and that is exactly beta reduction.

Böhm-tree readback is game semantics made operational. Probing a node
with `EntryPath + [λ0×pad]` returns

```
reverse(climb(μ)) + [λ0×l, λ1, @0×k, λ0×(pad−n)]
```

where `n` is the node's binder count, `k` its arity, `μ` the ancestor
whose chain binds its head at local position `l`, and `climb(μ)` — the
bounce's push-sequence ascending back to the root — is fully predictable
from already-decoded ancestors. `EntryPath` and `HeadPath` build
recursively exactly as the decoder walks the tree, so extraction is
deterministic and exact.

In [3]:
ROLES = {"@": (0, (2, 1)), "λ": (1, (0, 2))}
L0, L1, A0, A1 = ("λ", 0), ("λ", 1), ("@", 0), ("@", 1)


def logical_ports(cmap, index):
    ports = cmap._box_port_indices[index]
    arity = len(cmap.boxes[index].dom)
    return ports[:arity] + tuple(reversed(ports[arity:]))


def port_table(cmap):
    table = {}
    for index, box in enumerate(cmap.boxes):
        for wire, p in enumerate(logical_ports(cmap, index)):
            table[p] = (index, box.name, wire)
    return table


def run(cmap, stack, max_steps=5_000_000, boxes=None, watermark=False):
    """The machine; ``boxes`` maps constant names to strategies."""
    root, table = cmap.n_ports - 1, port_table(cmap)
    port, stack = cmap.edges[root], tuple(stack)
    low = len(stack)
    for _ in range(max_steps):
        low = min(low, len(stack))
        if port == root:
            return (stack, low) if watermark else stack
        if port not in table:
            return (None, low) if watermark else None
        index, name, wire = table[port]
        if name in ROLES:
            principal, auxes = ROLES[name]
            if wire == principal:
                if not stack:
                    return (None, 0) if watermark else None
                (kind, j), stack = stack[0], stack[1:]
                out_wire = auxes[j]
            else:
                stack = ((name, auxes.index(wire)), ) + stack
                out_wire = principal
        else:
            stack = boxes[name](stack)
            if stack is None:
                return (None, low) if watermark else None
            out_wire = wire
        port = cmap.edges[logical_ports(cmap, index)[out_wire]]
    return (None, low) if watermark else None


def strip_run(seq, symbol):
    count = 0
    while count < len(seq) and seq[count] == symbol:
        count += 1
    return count, seq[count:]


def parse_own(rest, pad):
    local, rest = strip_run(rest, L0)
    if not rest or rest[0] != L1:
        return None
    arity, rest = strip_run(rest[1:], A0)
    tail, rest = strip_run(rest, L0)
    return None if rest else (local, arity, pad - tail)


def extract(cmap, pad=12, max_nodes=100, max_steps=5_000_000, boxes=None):
    """Exact Böhm-tree readback: {address: (binders, head, arity)}."""
    nodes, entry_paths, head_paths = {}, {}, {}
    scopes, level_of, mu_of = {}, {}, {}

    def climb(node):
        if node == ():
            return ()
        parent = node[:-1]
        hops = (A1, ) + (A0, ) * node[-1] + (L1, )
        local = level_of[parent] - scopes[mu_of[parent][:-1]]\
            if mu_of[parent] else level_of[parent]
        return hops + (L0, ) * local + climb(mu_of[parent])

    todo = [((), ())]
    while todo and len(nodes) < max_nodes:
        address, entry = todo.pop(0)
        response = run(cmap, entry + (L0, ) * pad,
                       max_steps=max_steps, boxes=boxes)
        parsed = None
        if response is not None:
            candidates = sorted(
                [address[:d] for d in range(len(address), -1, -1)],
                key=lambda mu: -len(climb(mu)))
            for mu in candidates:
                junk = tuple(reversed(climb(mu)))
                own = parse_own(response[len(junk):], pad)\
                    if response[:len(junk)] == junk else None
                if own is not None:
                    parsed = (mu, ) + own
                    break
        if parsed is None:
            nodes[address] = None
            continue
        mu, local, arity, n_binders = parsed
        scope = scopes[address[:-1]] if address else 0
        scopes[address] = scope + n_binders
        level = (scopes[mu[:-1]] if mu else 0) + local
        nodes[address] = (n_binders, level, arity)
        level_of[address], mu_of[address] = level, mu
        entry_paths[address] = entry
        head_paths[address] = entry_paths[mu] + (L0, ) * local + (L1, )
        for j in range(arity):
            todo.append((address + (j, ),
                         head_paths[address] + (A0, ) * j + (A1, )))
    return nodes


def tree_nodes(term):
    result = {}

    def walk(t, address):
        result[address] = (len(t.variables), t.head, len(t.args))
        for j, arg in enumerate(t.args):
            walk(arg, address + (j, ))
    walk(tree(term), ())
    return result

In [4]:
def random_linear_term(size, rng, retries=2000):
    counter = [0]

    def fresh():
        counter[0] += 1
        return Variable(f"v{counter[0]}", o)

    def sample(size, free):
        if size == 0:
            return free[0] if len(free) == 1 else None
        if not free or rng.random() < 0.45:
            var = fresh()
            body = sample(size - 1, free + [var])
            return None if body is None else Abstraction(var, body)
        rng.shuffle(free)
        cut = rng.randint(0, len(free))
        split = rng.randint(0, size - 1)
        func = sample(split, free[:cut])
        if func is None:
            return None
        args = sample(size - split - 1, free[cut:])
        return None if args is None else func(args)

    for _ in range(retries):
        term = sample(size, [])
        if term is not None:
            return term
    raise ValueError


rng = random.Random(0)
suite = [true, false, not_(true), gid(true), copy(true), copy(false),
         pair(true)(false), xor_ref(bits[1])(bits[0])]
suite += [random_linear_term(2 * rng.randint(1, 5) + 1, rng)
          for _ in range(20)]
exact = sum(extract(term.to_map()) == tree_nodes(term) for term in suite)
print(f"exact Böhm-tree readback, zero parameters: {exact}/{len(suite)}")

exact Böhm-tree readback, zero parameters: 28/28


The same machine vectorizes to JAX: stacks become one-hot arrays
`(L, D)`, routing multiplies by the popped head's channels (exact on
one-hots, differentiable in general), and `lax.scan` over rounds jit-fuses
the whole map into one XLA program — the QNLP trick applied to the
execution formula. We keep the discrete machine below because it is the
faster harness for *this* notebook's supervised pipeline; the JAX machine
is what end-to-end gradient experiments plug into.

In [5]:
L, D = 48, 4
SYM = {L0: 0, L1: 1, A0: 2, A1: 3}


def push_v(stack, symbol):
    top = jnp.broadcast_to(symbol, stack.shape[:-2] + (1, D))
    return jnp.concatenate([top, stack[..., :-1, :]], axis=-2)


def pop_v(stack):
    zero = jnp.zeros_like(stack[..., :1, :])
    return stack[..., 0, :], jnp.concatenate(
        [stack[..., 1:, :], zero], axis=-2)


def gamma_step(kind, x_p, a_p, x_auxes, a_auxes):
    sym = [jnp.eye(D)[SYM[(kind, j)]] for j in range(2)]
    out_p = sum(a[..., None, None] * push_v(x, s)
                for x, a, s in zip(x_auxes, a_auxes, sym))
    head, tail = pop_v(x_p)
    outs, acts = [], []
    for j in range(2):
        weight = a_p * (head[..., SYM[("λ", j)]] + head[..., SYM[("@", j)]])
        outs.append(weight[..., None, None] * tail)
        acts.append(weight)
    return out_p, sum(a_auxes), outs, acts


def machine(cmap, rounds):
    root = cmap.n_ports - 1
    entry = int(cmap.edges[root])
    boxes = [(box.name, logical_ports(cmap, index))
             for index, box in enumerate(cmap.boxes)]
    edges = np.array([cmap.edges[p] for p in range(cmap.n_ports)])
    n_ports = cmap.n_ports

    @jax.jit
    def respond(queries):
        q = queries.shape[0]
        stacks = jnp.zeros((q, n_ports, L, D)).at[:, entry].set(queries)
        acts = jnp.zeros((q, n_ports)).at[:, entry].set(1.)

        def body(state, _):
            stacks, acts = state
            out_s = [jnp.zeros_like(stacks[:, 0])] * n_ports
            out_a = [jnp.zeros_like(acts[:, 0])] * n_ports
            for name, ports in boxes:
                principal, auxes = ROLES[name]
                p, a0, a1 = (ports[principal],
                             ports[auxes[0]], ports[auxes[1]])
                o_p, oa_p, (o_0, o_1), (oa_0, oa_1) = gamma_step(
                    name, stacks[:, p], acts[:, p],
                    [stacks[:, a0], stacks[:, a1]],
                    [acts[:, a0], acts[:, a1]])
                for port, s, a in [(p, o_p, oa_p), (a0, o_0, oa_0),
                                   (a1, o_1, oa_1)]:
                    out_s[port] = out_s[port] + s
                    out_a[port] = out_a[port] + a
            outgoing = jnp.stack(out_s, axis=1)[:, edges]
            outgoing_a = jnp.stack(out_a, axis=1)[:, edges]
            outgoing = outgoing.at[:, root].add(stacks[:, root])
            outgoing_a = outgoing_a.at[:, root].add(acts[:, root])
            return (outgoing, outgoing_a), None

        (stacks, acts), _ = jax.lax.scan(
            body, (stacks, acts), None, length=rounds)
        return stacks[:, root], acts[:, root]
    return respond


def encode(symbols):
    stack = np.zeros((L, D))
    for row, s in enumerate(symbols):
        stack[row, SYM[s]] = 1.
    return stack


INV = {v: k for k, v in SYM.items()}


def decode_stack(stack, activity):
    if activity < 0.5:
        return None
    symbols = []
    for row in np.asarray(stack):
        if row.sum() < 0.5:
            break
        symbols.append(INV[int(row.argmax())])
    return tuple(symbols)


t0 = time.time()
term = copy(true)
respond = machine(term.to_map(), rounds=2048)
response, activity = respond(jnp.asarray(encode((L0, ) * 12))[None])
assert decode_stack(response[0], float(activity[0]))\
    == run(term.to_map(), (L0, ) * 12)
print(f"the jitted machine agrees with the discrete one "
      f"({time.time() - t0:.1f}s compile+run, "
      f"{len(term.to_map().boxes)} boxes, 2048 fused rounds)")

the jitted machine agrees with the discrete one (4.2s compile+run, 77 boxes, 2048 fused rounds)


## GoI compositionality: the box's job is its term's execution formula

A gate in a circuit map is a single univalent box. What must it do? By
compositionality of the execution formula, exactly what its reference
term's map does through its root port. So an **oracle box** that runs the
reference map on every incoming token makes the hybrid machine exact — and
every oracle call is a labelled training example ``stack_in -> stack_out``
for the learned box.

Two things make the dataset small and the learning easy:

* **the watermark**: tracking the minimum stack length during the oracle
  run gives the *true read depth* — rows below it ride through untouched,
  so a visit reduces to the rule ``in[:read] -> head`` with
  ``out = head + in[read:]``;
* **structural boxes**: `copy`, `pair` and `not` are boxed too. GoI paths
  grow exponentially through inlined wiring (the classical cost of the
  unmemoized interaction machine), while a box answers in one step.

The training circuits are truth-table applications, partial applications
with one open slot, and gate outputs under `not`/`pair`/`copy` and
two-gate compositions. Half adders and the gate terms themselves are held
out.

In [6]:
STRUCTURAL = {"copy": (o("copy"), copy), "pair": (o("pair"), pair),
              "not": (o("not"), not_)}
copy_box, pair_box, not_box = (c for c, _ in STRUCTURAL.values())
ORACLES = {**{name: ref.to_map()
              for name, (_, ref, _, _) in GATES.items()},
           **{name: ref.to_map() for name, (_, ref) in
              STRUCTURAL.items()}}
VISITS = {name: {} for name in ORACLES}


def oracle_box(name, record=True):
    def strategy(stack):
        answer, low = run(ORACLES[name], stack, watermark=True)
        if answer is not None and record:
            VISITS[name][stack] = (answer, len(stack) - low)
        return answer
    return strategy


ORACLE_BOXES = {name: oracle_box(name) for name in ORACLES}


def circuits():
    out = []
    for name, (c, r, arity, op) in GATES.items():
        for args in inputs_of(arity):
            terms = [bits[bit] for bit in args]
            out.append((f"{name}{args}", apply_all(c, terms),
                        apply_all(r, terms)))
        for slot in range(arity):
            for bit in (0, 1):
                filled = [bits[bit]] * (arity - 1)

                def build(head, slot=slot, filled=filled):
                    return o(lambda hole: apply_all(
                        head, filled[:slot] + [hole] + filled[slot:]))
                out.append((f"{name}[slot{slot}={bit}]",
                            build(c), build(r)))
    for name in ("xor", "and", "or"):
        c, r, _, op = GATES[name]
        for a in (0, 1):
            for b in (0, 1):
                ca, cb = bits[a], bits[b]
                out += [(f"not({name}{(a, b)})",
                         not_box(c(ca)(cb)), not_(r(ca)(cb))),
                        (f"pair({name}{(a, b)},{a})",
                         pair_box(c(ca)(cb))(ca), pair(r(ca)(cb))(ca)),
                        (f"copy({name}{(a, b)})",
                         copy_box(c(ca)(cb)), copy(r(ca)(cb)))]
    for a in (0, 1):
        out += [(f"not({a})", not_box(bits[a]), not_(bits[a])),
                (f"copy({a})", copy_box(bits[a]), copy(bits[a]))]
        for b in (0, 1):
            out.append((f"pair({a},{b})", pair_box(bits[a])(bits[b]),
                        pair(bits[a])(bits[b])))
    for n1, n2 in [("xor", "and"), ("and", "or"), ("or", "xor"),
                   ("nand", "xor"), ("nor", "and"), ("xor", "nand")]:
        c1, r1 = GATES[n1][0], GATES[n1][1]
        c2, r2 = GATES[n2][0], GATES[n2][1]
        for a in (0, 1):
            for b in (0, 1):
                for x in (0, 1):
                    out.append((f"{n1}({n2}{(a, b)},{x})",
                                c1(c2(bits[a])(bits[b]))(bits[x]),
                                r1(r2(bits[a])(bits[b]))(bits[x])))
    return out


t0 = time.time()
data = circuits()
exact = 0
for label, term, reference in data:
    got = extract(term.to_map(), boxes=ORACLE_BOXES)
    exact += got == tree_nodes(reference)
print(f"hybrid extraction with oracle boxes: {exact}/{len(data)} exact"
      f" ({time.time() - t0:.0f}s)")
for name in GATES:
    print(f"  {name}: {len(VISITS[name])} unique visits")

hybrid extraction with oracle boxes: 146/146 exact (541s)
  xor: 1213198 unique visits
  and: 46923 unique visits
  or: 39547 unique visits
  nand: 14256 unique visits
  nor: 352 unique visits
  mux: 414 unique visits


## From a million visits to a hundred rules

The watermark collapses the visit sets: rows below the read depth are
provably unread, so the strategies are tiny — this is the gate's *innocent
strategy* in game-semantics terms, and its finiteness is why the gates are
learnable at all.

In [7]:
def true_rules(pairs):
    rules = {}
    for stack_in, (stack_out, read) in pairs.items():
        key = stack_in[:read]
        head = stack_out[:len(stack_out) - (len(stack_in) - read)]
        assert stack_out == head + stack_in[read:]
        assert rules.get(key, head) == head, "rule clash"
        rules[key] = head
    return rules


RULES = {name: true_rules(VISITS[name]) for name in GATES}
for name in GATES:
    print(f"{name}: {len(VISITS[name])} visits"
          f" -> {len(RULES[name])} true rules")

xor: 1213198 visits -> 142 true rules
and: 46923 visits -> 39 true rules
or: 39547 visits -> 39 true rules
nand: 14256 visits -> 49 true rules
nor: 352 visits -> 46 true rules
mux: 414 visits -> 27 true rules


## Training the gates in JAX

Each gate is a two-layer MLP predicting ``(read depth, rewritten head)``
from a 16-row window; the machine appends the untouched tail itself.
Randomizing the rows below the read depth at training time teaches the
net that they are unread — the invariance that makes open-term traffic
(the eta probes below) decode correctly.

In [8]:
T, HEAD_ROWS, BLANK = 16, 22, D
N_CLASSES = D + 1


def grid(symbols, rows):
    out = np.full(rows, BLANK, dtype=np.int32)
    for row, s in enumerate(symbols[:rows]):
        out[row] = SYM[s]
    return out


def forward(params, x):
    h = jax.nn.one_hot(x, N_CLASSES).reshape(x.shape[0], -1)
    for depth, (w, b) in enumerate(params):
        h = h @ w + b
        if depth + 1 < len(params):
            h = jax.nn.relu(h)
    return h[:, :T + 1], h[:, T + 1:].reshape(-1, HEAD_ROWS, N_CLASSES)


def loss_fn(params, x, read, head):
    read_logits, head_logits = forward(params, x)
    ce_read = -jax.nn.log_softmax(read_logits)[
        jnp.arange(x.shape[0]), read]
    ce_head = -jnp.take_along_axis(
        jax.nn.log_softmax(head_logits, axis=-1),
        head[..., None], axis=-1)[..., 0]
    return ce_read.mean() + ce_head.mean() * HEAD_ROWS


def adam(params, grads, state, step, lr):
    m, v = state
    m = jax.tree.map(lambda a, g: 0.9 * a + 0.1 * g, m, grads)
    v = jax.tree.map(lambda a, g: 0.999 * a + 0.001 * g * g, v, grads)
    c1, c2 = 1 - 0.9 ** step, 1 - 0.999 ** step
    return jax.tree.map(
        lambda p, mm, vv: p - lr * (mm / c1)
        / (jnp.sqrt(vv / c2) + 1e-8), params, m, v), (m, v)


def train_gate(name, rules, key, hidden=256, steps=4000, batch=512):
    keys = sorted(rules)
    x = np.stack([grid(k, T) for k in keys])
    read = np.array([len(k) for k in keys], dtype=np.int32)
    head = np.stack([grid(rules[k], HEAD_ROWS) for k in keys])
    sizes = [T * N_CLASSES, hidden, hidden,
             (T + 1) + HEAD_ROWS * N_CLASSES]
    params = []
    for depth in range(len(sizes) - 1):
        key, sub = jax.random.split(key)
        params.append((
            jax.random.normal(sub, (sizes[depth], sizes[depth + 1]))
            * np.sqrt(2. / sizes[depth]), jnp.zeros(sizes[depth + 1])))
    state = (jax.tree.map(jnp.zeros_like, params),
             jax.tree.map(jnp.zeros_like, params))
    vg = jax.jit(jax.value_and_grad(loss_fn))
    gen, n, t0 = np.random.default_rng(0), x.shape[0], time.time()
    for step in range(1, steps + 1):
        idx = gen.integers(0, n, size=min(batch, n))
        noise = gen.integers(0, N_CLASSES, size=(len(idx), T))
        x_aug = np.where(np.arange(T)[None, :] < read[idx][:, None],
                         x[idx], noise).astype(np.int32)
        value, grads = vg(params, jnp.asarray(x_aug),
                          jnp.asarray(read[idx]), jnp.asarray(head[idx]))
        params, state = adam(params, grads, state, step,
                             1e-3 if step < steps * 0.8 else 2e-4)
    r_logits, h_logits = forward(params, jnp.asarray(x))
    acc = float(((np.asarray(r_logits.argmax(-1)) == read)
                 & (np.asarray(h_logits.argmax(-1)) == head)
                 .all(axis=1)).mean())
    print(f"{name}: {n} rules, rule-accuracy {acc:.4f},"
          f" final loss {float(value):.4f} ({time.time() - t0:.0f}s)")
    return params


key = jax.random.PRNGKey(0)
NETS = {}
for name in GATES:
    key, sub = jax.random.split(key)
    NETS[name] = train_gate(name, RULES[name], sub,
                            hidden=512 if name == "xor" else 256,
                            steps=6000 if name == "xor" else 3000)

xor: 142 rules, rule-accuracy 1.0000, final loss 0.0004 (54s)


and: 39 rules, rule-accuracy 1.0000, final loss 0.0019 (14s)


or: 39 rules, rule-accuracy 1.0000, final loss 0.0013 (11s)


nand: 49 rules, rule-accuracy 1.0000, final loss 0.0014 (13s)


nor: 46 rules, rule-accuracy 1.0000, final loss 0.0018 (13s)


mux: 27 rules, rule-accuracy 1.0000, final loss 0.0007 (13s)


## The scorecard: everything decodes exactly

The learned boxes replace the oracles; `copy`/`pair`/`not` stay exact
library strategies. Held out: the half adders (circuits never seen in
training) and the gate terms themselves — probing `λp.λq.gate(p)(q)`
reads the Mairson term back out of the trained weights, node by node,
through open-term traffic the boxes never saw.

In [9]:
def net_box(name):
    def strategy(stack):
        x = np.asarray(grid(stack, T))[None]
        read_logits, head_logits = forward(NETS[name], x)
        read = int(np.asarray(read_logits[0]).argmax())
        head = []
        for row in np.asarray(head_logits[0]).argmax(-1):
            if row == BLANK:
                break
            head.append(INV[int(row)])
        return tuple(head) + tuple(stack[read:])
    return strategy


LEARNED = {name: net_box(name) for name in GATES}
LEARNED.update({name: ORACLE_BOXES[name] for name in STRUCTURAL})


def decode_learned(term, reference, label):
    got = extract(term.to_map(), boxes=LEARNED, max_steps=20_000_000)
    ok = got == tree_nodes(reference)
    print(f"{label}: {'EXACT' if ok else 'MISS'}")
    return ok


print("=== truth tables ===")
total = 0
for name, (c, r, arity, op) in GATES.items():
    for args in inputs_of(arity):
        terms = [bits[bit] for bit in args]
        total += decode_learned(apply_all(c, terms),
                                apply_all(r, terms), f"{name}{args}")
print(f"-> {total}/28 rows exact")

=== truth tables ===


xor(0, 0): EXACT


xor(0, 1): EXACT


xor(1, 0): EXACT


xor(1, 1): EXACT


...

mux(0, 1, 1): EXACT


mux(1, 0, 0): EXACT


mux(1, 0, 1): EXACT


mux(1, 1, 0): EXACT


mux(1, 1, 1): EXACT
-> 28/28 rows exact


In [10]:
xor_c, and_c = GATES["xor"][0], GATES["and"][0]
half = o(lambda a: o(lambda b:
    copy_box(a)(o(lambda a1: o(lambda a2:
    copy_box(b)(o(lambda b1: o(lambda b2:
    pair_box(xor_c(a1)(b1))(and_c(a2)(b2))))))))))
half_r = o(lambda a: o(lambda b:
    copy(a)(o(lambda a1: o(lambda a2:
    copy(b)(o(lambda b1: o(lambda b2:
    pair(xor_ref(a1)(b1))(and_ref(a2)(b2))))))))))
print("=== held out: half adders ===")
for a in (0, 1):
    for b in (0, 1):
        decode_learned(half(bits[a])(bits[b]), half_r(bits[a])(bits[b]),
                       f"half_adder({a},{b})")
print("=== held out: reading the gate terms back from the weights ===")
for name, (c, r, arity, _) in GATES.items():
    if arity != 2:
        continue
    decode_learned(o(lambda p: o(lambda q: c(p)(q))),
                   o(lambda p: o(lambda q: r(p)(q))), f"eta({name})")

=== held out: half adders ===


half_adder(0,0): EXACT


half_adder(0,1): EXACT


half_adder(1,0): EXACT


half_adder(1,1): EXACT
=== held out: reading the gate terms back from the weights ===
eta(xor): EXACT
eta(and): EXACT
eta(or): EXACT
eta(nand): EXACT


eta(nor): EXACT


## What this shows, and what comes next

The v1 experiments on this branch learned everything end-to-end — shared
neural gamma/delta cores, a learned root — and plateaued: truth tables
were learnable, composition and term readback were not. Fixing the
machine and moving all parameters into the constants inverts every one of
those results, at a fraction of the compute: visits collapse to dozens of
rules per gate, training takes seconds per box, and truth tables, unseen
half adders and the gate terms themselves all decode exactly.

Two structural lessons travel beyond Booleans. First, GoI
compositionality turns end-to-end learning into supervised learning at
box interfaces — the oracle is the reference term's own execution
formula. Second, interaction length is the real cost: paths through
inlined wiring grow exponentially (the classical unmemoized-IAM
slowdown), and boxing a subterm compresses its internal paths to one
step — learned strategies double as accelerators of the token machine.

Next: the almost-linear fragment with **two stacks** (multiplicative and
exponential), where copying is a delta node of arbitrary arity, and a
synthetic dataset of **Church arithmetic** — constants `zero, one, two,
three, plus, times, square` — learned from equations between their
compositions.